In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import numpy as np
import pandas as pd

hour_features_df = pd.read_csv("dataset/elevator_ml_dataset.csv")

features = ['hour', 'weekday', 'demand_count', 'avg_floor', 'most_common_floor',
       'avg_direction', 'peak_hours']
target = "best_resting_floor"

X = hour_features_df[features]
y = hour_features_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
param_grid = {
    "n_estimators": [50, 100, 150],
    "max_depth": [None, 4, 8],
    "min_samples_split": [2, 4]
}
rf = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")
grid = GridSearchCV(rf, param_grid, cv=3, scoring="accuracy", verbose=1)
grid.fit(X_train, y_train)
print("Mejores parámetros:", grid.best_params_)

In [ ]:
import matplotlib.pyplot as plt
best_rf = grid.best_estimator_
y_pred = best_rf.predict(X_test)

print(classification_report(y_test, y_pred))

# Matriz de confusión
cm = ConfusionMatrixDisplay.from_estimator(
    best_rf, X_test, y_test, cmap="Blues", display_labels=sorted(y.unique())
)
plt.title("Matriz de Confusión: Piso de Descanso Óptimo")
plt.show()

In [ ]:
importances = best_rf.feature_importances_
feature_names = X.columns
indices = np.argsort(importances)[::-1]

plt.figure(figsize=(8,5))
plt.title("Importancia de Features")
plt.bar(range(len(importances)), importances[indices], align="center")
plt.xticks(range(len(importances)), [feature_names[i] for i in indices], rotation=45)
plt.show()

In [ ]:
import joblib

joblib.dump(best_rf, "/content/best_resting_floor_model.joblib")